####DAY 8 (27/02/26) – Batch Inference Pipeline
####🏗️ Architecture & Strategy
Welcome to Day 8! You have successfully built, tuned, and registered a production-ready model. Today, we put that model to work to generate actual business value. We are building a Batch Inference Pipeline.
Instead of making real-time predictions one by one (which is expensive and often unnecessary for eCommerce targeting), we will run a scheduled batch job.

####Our Strategy:

* **Dynamic Model Loading**: We will load our model directly from the Unity Catalog Model Registry using its ``champion`` alias. If we ever retrain and promote a new model, this code will automatically pick up the new version without a single code change.

* **Score the Entire Universe**: We will grab the full `silver_user_features` table (all users, not just our training sample) and pass it through our model.

* **Probability Extraction (Senior PySpark Trick)**: Spark ML outputs probabilities as a complex Vector (e.g.,`[0.85, 0.15]`). Business analysts using SQL cannot easily read Vectors. We will use a highly efficient PySpark function (`vector_to_array`) to extract the exact probability of purchase into a clean, readable decimal column.

* **Gold Layer Creation**: We will save these parsed predictions to a Gold Delta table.  We will apply `ZORDER BY` on the prediction probability to optimize the table so marketing teams can instantly query their top buyers.

####Load the Champion Model from Unity Catalog
We start by connecting to Unity Catalog and pulling the exact model we registered yesterday.

In [0]:
import os
import mlflow

# 1. Environment Setup
catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"
volume_name = "ml_assets"
mlflow_tmp_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/mlflow_staging"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

# Safety net: Force the cluster OS to use the UC Volume
os.environ["MLFLOW_DFS_TMP"] = mlflow_tmp_path

# 2. Define the Model URI using the 'champion' alias
uc_model_name = f"{catalog_name}.{schema_name}.purchase_prediction_classifier"
model_uri = f"models:/{uc_model_name}@champion"

print(f"🔄 Fetching the 'champion' model from Unity Catalog...")
print(f"   ➤ URI: {model_uri}")

# 3. Load the Model
# ⚠️ THE FIX: Pass the Unity Catalog Volume path so Spark has a secure place to unpack the model
loaded_champion_model = mlflow.spark.load_model(
    model_uri=model_uri,
    dfs_tmpdir=mlflow_tmp_path
)

print("✅ Model loaded successfully and is ready for inference.")

####Score All Users & Parse Probabilities
We bring in the full Silver feature table and run our batch scoring. Then, we transform the output into a clean, analyst-friendly format.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

# 1. Load the entire user base (Silver Layer)
print("⏳ Loading full user base from Silver layer...")
df_all_users = spark.table("silver_user_features")

# 2. Perform Batch Inference
print("🧠 Scoring all users using the Champion Model...")
raw_predictions = loaded_champion_model.transform(df_all_users)

# 3. Parse the Probability Vector
# Spark ML's 'probability' column is a dense vector: [probability_of_0, probability_of_1]
# We use vector_to_array to split it, and grab index 1 (the probability they WILL purchase)
print("🧹 Cleaning prediction output for business users...")
parsed_predictions = raw_predictions.withColumn(
    "purchase_intent_score", 
    F.round(vector_to_array(F.col("probability"))[1], 4) # Extract index 1, round to 4 decimals
).select(
    "user_id", 
    "total_events", 
    "view_count", 
    "cart_count", 
    "purchase_intent_score", 
    F.col("prediction").cast("integer").alias("predicted_to_buy")
)

display(parsed_predictions.limit(5))

####Save Predictions to Gold Layer
We write this clean, scored dataset to our Gold layer. We optimize the table specifically for the marketing team's use case: querying the users with the highest intent scores.

In [0]:
# ---------------------------------------------------------
# GOLD LAYER CREATION & OPTIMIZATION
# ---------------------------------------------------------
gold_table_name = "gold_predicted_buyers"
print(f"💾 Saving predictions to Gold Layer: {catalog_name}.{schema_name}.{gold_table_name}...")

# 1. Write to Delta (Overwrite mode ensures the daily batch job refreshes the table)
parsed_predictions.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)

# 2. Optimize and Z-Order
# The Marketing team will frequently query this table filtering for high 'purchase_intent_score'.
# Z-Ordering by this column ensures instantaneous query performance for those workloads.
print("🚀 Optimizing Gold table and Z-Ordering by 'purchase_intent_score'...")
optimize_metrics = spark.sql(f"OPTIMIZE {gold_table_name} ZORDER BY (purchase_intent_score)")

print("✅ Batch Inference Pipeline complete. Data is secured in the Gold Layer.")

###Identify Top Predicted Buyers
Let's put on our Data Analyst hats for a moment to prove the value of our pipeline to the business stakeholders. We will extract the top targets for an immediate marketing email campaign.

In [0]:
# ---------------------------------------------------------
# BUSINESS INSIGHTS: TOP PREDICTED BUYERS
# ---------------------------------------------------------
print("🎯 Extracting the Top 20 High-Intent Users for Marketing Campaign...")

# Query the Gold table for users who haven't bought yet, but have the highest probability of buying
top_buyers_df = spark.sql(f"""
    SELECT 
        user_id,
        cart_count,
        view_count,
        purchase_intent_score
    FROM {gold_table_name}
    ORDER BY purchase_intent_score DESC
    LIMIT 20
""")

display(top_buyers_df)

# 💡 INSTRUCTIONS FOR DATABRICKS NATIVE PLOTTING:
# You can click the '+' icon -> 'Visualization' to create a Bar Chart of these top users
# to visualize the correlation between their cart_count and their intent score!

####Key Learnings & Interview Talking Points
If a technical recruiter asks about how you deploy models for batch processing, highlight these critical concepts:

* **Dynamic Model Registry Fetching**: "In my batch inference pipelines, I never hardcode model versions or file paths. I leverage the MLflow Model Registry's aliasing system (@champion) to dynamically fetch the production-approved model. This completely decouples the Data Science training lifecycle from the Data Engineering scheduling lifecycle."

* **Handling Spark ML Outputs for BI**: "Spark ML algorithms natively output probabilities as complex Dense Vectors. I understand that downstream Data Analysts and BI tools (like Tableau or PowerBI) struggle with vectors. Therefore, I proactively use PySpark's vector_to_array function to extract the positive class probability into a clean, queryable decimal column before persisting to the Gold layer."

* **Workload-Specific Z-Ordering**: "When writing the final predictions to the Delta Lake Gold layer, I anticipated how the data would be consumed. Since marketing teams will constantly query WHERE purchase_intent_score > 0.8, I applied ZORDER BY (purchase_intent_score) to physically cluster high-intent users on disk, drastically reducing query I/O latency."

* **PySpark Pipelines in Action**: "Because I logged my model as a holistic PySpark Pipeline on Day 7, my inference code is incredibly clean. I didn't need to rewrite any VectorAssembler logic in the batch job; the model inherently knew how to preprocess the raw silver table data.